###### Imports and Pulls

In [1]:
import pandas as pd
import numpy as np
import requests
#import io
import pickle
from collections import deque
from functools import reduce
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 150)
pd.options.mode.chained_assignment = None  # default='warn'

###### API Key

In [2]:
#to read in... rb is read bite
with open('api_keys.pkl', 'rb') as keys_file:
        keys_dict_2 = pickle.load(keys_file)
#create a variable that contains your api key
api_key = keys_dict_2['CENSUS']

###### Geography Lists

In [3]:
GNRC = ['161', #Stewart
       '125', #Montgomery
       '083', #Houston
       '085', #Humphreys
       '043', #Dickson
       '021', #Cheatham
       '147', #Robertson
       '165', #Sumner
       '037', #Davidson
       '189', #Wilson
       '169', #Trousdale
       '187', #Williamson
       '149', #Rutherford
       '119'] #Maury

In [4]:
slaces = ['02180', #Ashland City: Cheatham
          '39660', #Kingston Springs: Cheatham
          '57480', #Pegram: Cheatham
          '59560', #Pleasant View: Cheatham
          '04620', #Belle Meade: Davidson
          '05140', #Berry Hill: Davidson
          '27020', #Forest Hills: Davidson
          '29920', #Goodlettsville: Davidson/Sumner
          '52006', #Nashville-Davidson metropolitan government (balance): Davidson
          '54780', #Oak Hill: Davidson
          '63140', #Ridgetop: Davidson/Robertson
          '09880', #Burns: Dickson
          '13080', #Charlotte: Dickson
          '20620', #Dickson: Dickson
          '69080', #Slayden: Dickson
          '76860', #Vanleer: Dickson
          '79980', #White Bluff: Dickson
          '24320', #Erin: Houston
          '73460', #Tennessee Ridge: Houston/Stewart
          '44840', #McEwen: Humphreys
          '52820', #New Johnsonville: Humphreys
          '78560', #Waverly: Humphreys
          '16540', #Columbia: Maury
          '51080', #Mount Pleasant: Maury
          '70580', #Spring Hill: Maury/Williamson
          '15160', #Clarksville: Montgomery
          '00200', #Adams: Robertson
          '11980', #Cedar Hill: Robertson
          '16980', #Coopertown: Robertson
          '18420', #Cross Plains: Robertson
          '30960', #Greenbrier: Robertson
          '48980', #Millersville: Robertson/Sumner
          '60280', #Portland: Robertson/Sumner
          '70500', #Springfield: Robertson
          '80200', #White House: Robertson/Sumner
          '22360', #Eagleville: Rutherford
          '41200', #La Vergne: Rutherford
          '51560', #Murfreesboro: Rutherford
          '69420', #Smyrna: Rutherford
          '18820', #Cumberland City: Stewart
          '21400', #Dover: Stewart
          '28540', #Gallatin: Sumner
          '33280', #Hendersonville: Sumner
          '79420', #Westmoreland: Sumner
          '08280', #Brentwood: Williamson
          '25440', #Fairview: Williamson
          '27740', #Franklin: Williamson
          '53460', #Nolensville: Williamson
          '73900', #Thompson's Station: Williamson
          '41520', #Lebanon: Wilson
          '50780', #Mount Juliet: Wilson
          '78320'] #Watertown: Wilson

At this time the only variable from the subject tables is "average commute time to work" so there is no data guide. In 2010 and 2020 the variable remains consistent: S0801_C01_046E

In [25]:
#2010 ACS 5 Year Subject Table Estimate
#counties
url_str= 'https://api.census.gov/data/2010/acs/acs5/subject?key='+api_key
predicates= {}
get_vars= ["NAME", 'S0801_C01_046E'] #the only variable is average commute
predicates["get"]= ",". join(get_vars)
predicates["for"]= "county:*"
predicates["in"]= "state:47" 
data = requests.get(url_str, params= predicates)
col_names = ['NAME', 'Mean Travel Time to Work', 'StateFIPS', 'GeoFIPS']
df = pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
df = df.loc[df['GeoFIPS'].isin(GNRC)]
#places
url_str= 'https://api.census.gov/data/2010/acs/acs5/subject?key='+api_key
predicates= {}
get_vars= ["NAME", 'S0801_C01_046E'] #the only variable is average commute
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47" 
data = requests.get(url_str, params= predicates)
col_names = ['NAME', 'Mean Travel Time to Work', 'StateFIPS', 'GeoFIPS']
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GeoFIPS'].isin(slaces)]
df = df.append(places).reset_index(drop = True)
#state call
url_str= 'https://api.census.gov/data/2010/acs/acs5/subject?key='+api_key
predicates= {}
get_vars= ["NAME", 'S0801_C01_046E'] #the only variable is average commute
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = ['NAME', 'Mean Travel Time to Work', 'StateFIPS']
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
df = df.append(state, ignore_index = True)
twentyten = df
print('Okay Finished')

Okay Finished


In [26]:
twentyten.tail()

,NAME,Mean Travel Time to Work,StateFIPS,GeoFIPS
62,"Waverly city, Tennessee",19.8,47,78560
63,"Westmoreland town, Tennessee",33.4,47,79420
64,"White Bluff town, Tennessee",32.6,47,79980
65,"White House city, Tennessee",28.8,47,80200
66,Tennessee,23.9,47,0


In [30]:
#2020 ACS 5 Year Subject Table Estimate
#counties
url_str= 'https://api.census.gov/data/2020/acs/acs5/subject?key='+api_key
predicates= {}
get_vars= ["NAME", 'S0801_C01_046E'] #the only variable is average commute
predicates["get"]= ",". join(get_vars)
predicates["for"]= "county:*"
predicates["in"]= "state:47" 
data = requests.get(url_str, params= predicates)
col_names = ['NAME', 'Mean Travel Time to Work', 'StateFIPS', 'GeoFIPS']
df = pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
df = df.loc[df['GeoFIPS'].isin(GNRC)]
#places
url_str= 'https://api.census.gov/data/2020/acs/acs5/subject?key='+api_key
predicates= {}
get_vars= ["NAME", 'S0801_C01_046E'] #the only variable is average commute
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47" 
data = requests.get(url_str, params= predicates)
col_names = ['NAME', 'Mean Travel Time to Work', 'StateFIPS', 'GeoFIPS']
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GeoFIPS'].isin(slaces)]
df = df.append(places).reset_index(drop = True)
#state call
url_str= 'https://api.census.gov/data/2020/acs/acs5/subject?key='+api_key
predicates= {}
get_vars= ["NAME", 'S0801_C01_046E'] #the only variable is average commute
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = ['NAME', 'Mean Travel Time to Work', 'StateFIPS']
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
df = df.append(state, ignore_index = True)
twentytwenty = df
print('Okay Finished')

Okay Finished


In [31]:
twentytwenty.tail()

,NAME,Mean Travel Time to Work,StateFIPS,GeoFIPS
62,"Waverly city, Tennessee",29.6,47,78560
63,"Westmoreland town, Tennessee",33.4,47,79420
64,"White Bluff town, Tennessee",38.7,47,79980
65,"White House city, Tennessee",38.1,47,80200
66,Tennessee,25.4,47,0


In [27]:
twentyten = twentyten.drop(columns = ['StateFIPS', 'GeoFIPS'])
twentyten = twentyten.set_index('NAME')
twentyten = twentyten.add_suffix(' 2010')
twentyten = twentyten.reset_index()

In [32]:
twentytwenty = twentytwenty.drop(columns = ['StateFIPS', 'GeoFIPS'])
twentytwenty = twentytwenty.set_index('NAME')
twentytwenty = twentytwenty.add_suffix(' 2020')
twentytwenty = twentytwenty.reset_index()

In [28]:
twentyten.head()

,NAME,Mean Travel Time to Work 2010
0,"Stewart County, Tennessee",32.0
1,"Sumner County, Tennessee",27.6
2,"Trousdale County, Tennessee",31.9
3,"Williamson County, Tennessee",27.0
4,"Wilson County, Tennessee",27.8


In [33]:
twentytwenty.head()

,NAME,Mean Travel Time to Work 2020
0,"Montgomery County, Tennessee",26.2
1,"Rutherford County, Tennessee",28.8
2,"Sumner County, Tennessee",29.5
3,"Trousdale County, Tennessee",41.0
4,"Williamson County, Tennessee",27.8


In [40]:
twentyten.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67 entries, 0 to 66
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   NAME                           67 non-null     object
 1   Mean Travel Time to Work 2010  67 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB


In [38]:
twentyten.to_csv('../../data/Prefix2010Subject.csv', index = False)

In [39]:
twentytwenty.to_csv('../../data/Prefix2020Subject.csv', index = False)